# Lancio di un training con le nuove specs

In [1]:
from smash_rl import train
from smash_rl.runs import launch, list_runs, status, stop
import subprocess
import ipywidgets as widgets

## Debug pre run

In [2]:
# lancio di pytest per verificare sia tutto a posto prima di addestrare
subprocess.run(["pytest", "-q", "--disable-warnings", "--tb=short"], check=True)

........................................................................ [ 46%]
........................................................................ [ 92%]
............                                                             [100%]
156 passed, 6 deselected in 7.07s


CompletedProcess(args=['pytest', '-q', '--disable-warnings', '--tb=short'], returncode=0)

In [17]:
for run in list_runs():
    for key, value in status(run).items():
        print(f"{key}: {value}")
    print("\n")

run_id: 55
run_name: rv4_cpu3_to5
pid: 1751903
pid_create_time: 1784021992.89
instance_base: 0
n_envs: 10
config_path: /home/luca/Desktop/Code/smash/runs/rv4_cpu3_to5/config.json
log_path: /home/luca/Desktop/Code/smash/runs/rv4_cpu3_to5/train.log
started_at: 2026-07-14T11:39:53
status: running
algo: dqn
total_steps: 6000000
alive: True
last_log_line: 39:58:308 Core/HW/EXI_DeviceSlippi.cpp:501 W[SLIPPI]: Creating file write thread...




## Run

In [15]:
# iperparametri di partenza per il training. Adattati da rl-zoo per atari, con qualche osservazione raccolta nei training preliminari

n_steps = int(6*1e6)   # facciamo 1.5 milioni di passi di addestramento
learning_rate = int(1e-4)    # per ora. Deve essere piccolo e verrà ottimizzato con optuna
buffer_size = int(0.3 * n_steps)  # 300k esperienze nel buffer, circa il 20% del totale, dovrebbe includere sia esperienze recenti che più vecchie senza privilegiare troppo nessuna delle due
learning_starts = 50_000  #round(0.03 * n_steps)  # iniziamo a imparare dopo il 3% dei passi, così da avere un buffer iniziale di esperienze
batch_size = 128    # aumentiamo la batch size per più stabilità nei gradient steps (il Q-learning tende a essere poco stabile)
gradient_steps = -1  # facciamo un gradient step per ogni passo di addestramento, così da sfruttare al massimo le esperienze raccolte
UTD = 0.25  # preso da rl-zoo per atari
train_freq = 4
n_envs = 10  # usiamo 10 ambienti in parallelo per raccogliere esperienze più velocemente
gradient_steps = round(UTD * n_envs * train_freq)  # calcoliamo il numero di gradient steps per avere un UTD di circa 0.25
target_update_interval = 10_000 # anche questo adattato da rl-zoo
exploration_initial_eps = 1.0  # iniziamo con eps=1.0 per esplorare molto all'inizio
exploration_final_eps = 0.02    # lasciamo un po' di esplorazione anche alla fine
exploration_fraction = 0.2      # l'esplorazione dura il 20% del training
gamma = 0.99    # questo dipende dal problema specifico, andrà adattato a parte


In [16]:
env_config = train.env_config(
    agent_char="MARIO",
    opp_char="LUIGI",
    stage="FINAL_DESTINATION",
    observation_function="full_obs",
    reward_function="v4",
    action_function="a_b",
    opp_level=5
)

dqn_config = train.DQN_config(
    exploration_initial_eps=exploration_initial_eps,
    exploration_final_eps=exploration_final_eps,
    exploration_fraction=exploration_fraction,
    buffer_size=buffer_size,
    learning_starts=learning_starts,
    batch_size=batch_size,
    gradient_steps=gradient_steps,
    target_update_interval=target_update_interval,
    gamma=gamma,
    train_freq=train_freq,
)

reset_exploration = {
    "initial_eps": 0.3,
    "final_eps": 0.05,
    "fraction": 0.2
}

train_config = train.TrainConfig(
    run_name="rv4_cpu3_to5",   # nome nuovo: non sovrascrive i checkpoint della run finita
    algo="dqn",
    total_steps=n_steps,
    env_kwargs=env_config,
    algo_kwargs=dqn_config,
    n_envs=n_envs,
    pretrained_path="checkpoints/rv4_cpu3_continuation/final.zip",
    reset_exploration=reset_exploration
)

launch(cfg=train_config)

rv4_cpu3_to5: pid 1751903, istanze [0, 10), porte 51441-51450, log /home/luca/Desktop/Code/smash/runs/rv4_cpu3_to5/train.log


RunHandle(run_name='rv4_cpu3_to5', run_id=55, pid=1751903, log_path=PosixPath('/home/luca/Desktop/Code/smash/runs/rv4_cpu3_to5/train.log'), config_path=PosixPath('/home/luca/Desktop/Code/smash/runs/rv4_cpu3_to5/config.json'))

## Stop

In [6]:
run_selector = widgets.Dropdown(
    options=[i["run_name"] for i in list_runs()],
    description='Select Run:',
    disabled=False,
)
display(run_selector)

Dropdown(description='Select Run:', options=('rv3_cpu3',), value='rv3_cpu3')

In [7]:
stop(run_selector.value)

rv3_cpu3: fermato


rv3_cpu3: non risponde a SIGTERM, SIGKILL al process group
